In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change "Questionnnair" -> "Questionnaire"
# if your folder uses the correct spelling.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HIQ_I.xpt"
csv_file = DATA_DIR / "HIQ_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 HIQ_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HIQ011",
    "HIQ031A",
    "HIQ031B",
    "HIQ031C",
    "HIQ031D",
    "HIQ031E",
    "HIQ031F",
    "HIQ031G",
    "HIQ031H",
    "HIQ031I",
    "HIQ031J",
    "HIQ031AA",
    "HIQ260",
    "HIQ105",
    "HIQ270",
    "HIQ210"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HIQ_I column names/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 9971:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 17:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,971 rows x 17 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached HIQ_I:
#
#     Genuine decimal variables = NONE
#     String variables          = NONE
#
#
# All 17 variables contain only whole-number:
#
#     respondent IDs
#     yes/no response codes
#     health-plan category codes
#
#
# Therefore all 17 can safely become nullable Int64.
#
#
# IMPORTANT:
#
# This rule is specific to HIQ_I.
#
# It does NOT mean genuine decimal variables in other
# NHANES files should be converted to integers.
#
# Continue preserving:
#
#     Dietary decimals
#     laboratory continuous results
#     Examination measurements
#     survey/sample weights
#
# exactly AS IS.
#

DECIMAL_COLUMNS = []

STRING_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 17:

    raise ValueError(
        "Expected exactly 17 whole-number/code fields."
    )


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "String fields:",
    STRING_COLUMNS
)

print(
    "Whole-number/code fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of zero:
#
#     5.397605346934028e-79
#
#
# Attached HIQ_I contains NONE.
#
# Therefore:
#
#     DO NOT perform tiny -> zero replacement.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 17 HIQ_I variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in HIQ_I."
    )


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected ordinary numeric zeros "
        "found in HIQ_I."
    )


print(
    "PASS: HIQ_I contains 0 tiny artifacts "
    "and 0 ordinary numeric zeros."
)


# ============================================================
# 6. NO ZERO RESTORATION REQUIRED
# ============================================================

#
# No XPORT zero corruption exists in this attachment.
#
# Therefore the original XPT itself is the exact
# numeric fidelity reference.
#

expected_df = source_df.copy()


print(
    "\nPASS: No tiny-to-zero restoration required."
)


# ============================================================
# 7. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is used ONLY for diagnosis.
#
# NO source values are modified.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


fractional_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal variables "
        "found in HIQ_I.\n\n"
        "These values will NOT be rounded:\n"
        + str(
            fractional_columns
        )
    )


print(
    "\nPASS: No genuine decimal variables "
    "found in HIQ_I."
)


# ============================================================
# 8. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 9. VALIDATE ALL FIELDS ARE WHOLE-NUMBER
# ============================================================

#
# Prove each populated observation is whole-number
# before doing Int64 conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


print(
    "PASS: All 17 HIQ_I variables are "
    "whole-number/code fields."
)


# ============================================================
# 10. CONVERT ALL FIELDS TO NULLABLE Int64
# ============================================================

#
# NO .round() during actual conversion.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 17 HIQ_I variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 11. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 12. CDC CHECK - HIQ011
# ============================================================

#
# Covered by health insurance:
#
#     1 = Yes          8,640
#     2 = No           1,312
#     7 = Refused          4
#     9 = Don't know      15
#
# Missing = 0
#

check_frequency(
    "HIQ011",
    {
        1: 8640,
        2: 1312,
        7: 4,
        9: 15
    },
    0
)


# ============================================================
# 13. CDC CHECK - HIQ031A
# ============================================================

#
# Private insurance:
#
#     14 = Covered by private insurance   4,424
#     77 = Refused                            4
#     99 = Don't know                        81
#
# Missing = 5,462
#

check_frequency(
    "HIQ031A",
    {
        14: 4424,
        77: 4,
        99: 81
    },
    5462
)


# ============================================================
# 14. CDC CHECK - HIQ031B
# ============================================================

check_frequency(
    "HIQ031B",
    {
        15: 1371
    },
    8600
)


# ============================================================
# 15. CDC CHECK - HIQ031C
# ============================================================

check_frequency(
    "HIQ031C",
    {
        16: 48
    },
    9923
)


# ============================================================
# 16. CDC CHECK - HIQ031D
# ============================================================

check_frequency(
    "HIQ031D",
    {
        17: 2384
    },
    7587
)


# ============================================================
# 17. CDC CHECK - HIQ031E
# ============================================================

check_frequency(
    "HIQ031E",
    {
        18: 97
    },
    9874
)


# ============================================================
# 18. CDC CHECK - HIQ031F
# ============================================================

check_frequency(
    "HIQ031F",
    {
        19: 331
    },
    9640
)


# ============================================================
# 19. CDC CHECK - HIQ031G
# ============================================================

check_frequency(
    "HIQ031G",
    {
        20: 6
    },
    9965
)


# ============================================================
# 20. CDC CHECK - HIQ031H
# ============================================================

check_frequency(
    "HIQ031H",
    {
        21: 715
    },
    9256
)


# ============================================================
# 21. CDC CHECK - HIQ031I
# ============================================================

check_frequency(
    "HIQ031I",
    {
        22: 366
    },
    9605
)


# ============================================================
# 22. CDC CHECK - HIQ031J
# ============================================================

check_frequency(
    "HIQ031J",
    {
        23: 773
    },
    9198
)


# ============================================================
# 23. CDC CHECK - HIQ031AA
# ============================================================

#
# 40 = No coverage of any type
#
# This is a multi-response code and therefore most
# observations are missing.
#

check_frequency(
    "HIQ031AA",
    {
        40: 4
    },
    9967
)


# ============================================================
# 24. HIQ011 / HIQ031 MULTI-RESPONSE ROUTING CHECK
# ============================================================

#
# HIQ031A through HIQ031AA are insurance-type
# follow-up fields.
#
# In the attached file:
#
#     Every HIQ011=1 participant has at least
#     one HIQ031 response.
#
#     No HIQ031 response occurs outside HIQ011=1.
#

HIQ031_COLUMNS = [
    "HIQ031A",
    "HIQ031B",
    "HIQ031C",
    "HIQ031D",
    "HIQ031E",
    "HIQ031F",
    "HIQ031G",
    "HIQ031H",
    "HIQ031I",
    "HIQ031J",
    "HIQ031AA"
]


hiq011_yes = (
    df["HIQ011"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


any_hiq031 = (
    df[HIQ031_COLUMNS]
    .notna()
    .any(axis=1)
    .to_numpy(
        dtype=bool
    )
)


routing_mismatches = int(
    np.sum(
        hiq011_yes
        !=
        any_hiq031
    )
)


print(
    "\n--- HIQ011 / HIQ031 ROUTING ---"
)

print(
    "HIQ011=1:",
    int(hiq011_yes.sum())
)

print(
    "Participants with at least one HIQ031 response:",
    int(any_hiq031.sum())
)

print(
    "Routing mismatches:",
    routing_mismatches
)


if routing_mismatches != 0:

    raise ValueError(
        "HIQ031 insurance-type availability "
        "does not match HIQ011=1."
    )


print(
    "PASS: HIQ011/HIQ031 routing validated."
)


# ============================================================
# 25. CHECK EACH HIQ031 FIELD OCCURS ONLY AMONG INSURED
# ============================================================

print(
    "\n--- HIQ031 INSURED-ONLY CHECK ---"
)


for col in HIQ031_COLUMNS:

    outside_insured = int(
        (
            df[col].notna()
            &
            ~df["HIQ011"].eq(1)
        )
        .sum()
    )


    print(
        f"{col}: outside HIQ011=1 ->",
        outside_insured
    )


    if outside_insured != 0:

        raise ValueError(
            f"{col} unexpectedly populated "
            "outside HIQ011=1."
        )


print(
    "PASS: All HIQ031 variables occur only "
    "among HIQ011=1 records."
)


# ============================================================
# 26. CDC CHECK - HIQ260
# ============================================================

#
# Have Medicare?
#
#     1 = Yes          101
#     2 = No           104
#     7 = Refused        0
#     9 = Don't know     7
#
# Missing = 9,759
#
#
# Eligibility includes age logic that is not stored
# in HIQ_I itself, so we validate frequencies but do
# NOT invent an age-based routing check here.
#

check_frequency(
    "HIQ260",
    {
        1: 101,
        2: 104,
        9: 7
    },
    9759
)


# ============================================================
# 27. CDC CHECK - HIQ105
# ============================================================

#
# Medicare insurance card available:
#
#     1 = Yes    966
#     2 = No      38
#
# Missing = 8,967
#

check_frequency(
    "HIQ105",
    {
        1: 966,
        2: 38
    },
    8967
)


# ============================================================
# 28. CDC CHECK - HIQ270
# ============================================================

#
# Do health plans cover prescriptions?
#
#     1 = Yes          8,147
#     2 = No             383
#     7 = Refused          3
#     9 = Don't know     117
#
# Missing = 1,321
#

check_frequency(
    "HIQ270",
    {
        1: 8147,
        2: 383,
        7: 3,
        9: 117
    },
    1321
)


# ============================================================
# 29. CDC CHECK - HIQ210
# ============================================================

#
# Any time without insurance in past 12 months?
#
#     1 = Yes            674
#     2 = No           7,965
#     7 = Refused          0
#     9 = Don't know      11
#
# Missing = 1,321
#

check_frequency(
    "HIQ210",
    {
        1: 674,
        2: 7965,
        9: 11
    },
    1321
)


# ============================================================
# 30. HIQ270 / HIQ210 ELIGIBILITY ROUTING
# ============================================================

#
# CDC's HIQ269 check item states:
#
# Continue if:
#
#     (HIQ011 = 1
#      AND HIQ031AA != 40)
#
# OR:
#
#     HIQ260 = 1
#
#
# Otherwise end the section.
#
#
# HIQ031AA is a multi-response field.
#
# Missing HIQ031AA means code 40 ("no coverage")
# was NOT selected.
#
# Therefore:
#
#     no_coverage = HIQ031AA == 40
#
#     eligible =
#         (HIQ011 == 1 and NOT no_coverage)
#         OR
#         HIQ260 == 1
#

no_coverage = (
    df["HIQ031AA"]
    .eq(40)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


hiq011_yes_mask = (
    df["HIQ011"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


hiq260_yes_mask = (
    df["HIQ260"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


followup_eligible = (
    (
        hiq011_yes_mask
        &
        ~no_coverage
    )
    |
    hiq260_yes_mask
)


hiq270_present = (
    df["HIQ270"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


hiq210_present = (
    df["HIQ210"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


hiq270_routing_mismatches = int(
    np.sum(
        followup_eligible
        !=
        hiq270_present
    )
)


hiq210_routing_mismatches = int(
    np.sum(
        followup_eligible
        !=
        hiq210_present
    )
)


print(
    "\n--- HIQ270 / HIQ210 ROUTING ---"
)

print(
    "Expected eligible records:",
    int(
        followup_eligible.sum()
    )
)

print(
    "HIQ270 populated:",
    int(
        hiq270_present.sum()
    )
)

print(
    "HIQ210 populated:",
    int(
        hiq210_present.sum()
    )
)

print(
    "HIQ270 routing mismatches:",
    hiq270_routing_mismatches
)

print(
    "HIQ210 routing mismatches:",
    hiq210_routing_mismatches
)


if int(
    followup_eligible.sum()
) != 8650:

    raise ValueError(
        "Unexpected HIQ follow-up eligibility count."
    )


if hiq270_routing_mismatches != 0:

    raise ValueError(
        "HIQ270 availability differs from "
        "CDC HIQ269 routing."
    )


if hiq210_routing_mismatches != 0:

    raise ValueError(
        "HIQ210 availability differs from "
        "CDC HIQ269 routing."
    )


print(
    "PASS: HIQ270 and HIQ210 routing "
    "validated exactly."
)


# ============================================================
# 31. HIQ270 / HIQ210 MISSING MASK CHECK
# ============================================================

#
# Both follow-up fields should be available for the
# same 8,650 participants.
#

hiq270_missing = (
    df["HIQ270"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


hiq210_missing = (
    df["HIQ210"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


mask_mismatches = int(
    np.sum(
        hiq270_missing
        !=
        hiq210_missing
    )
)


print(
    "\n--- HIQ270 / HIQ210 MISSING MASK ---"
)

print(
    "Missing-mask mismatches:",
    mask_mismatches
)


if mask_mismatches != 0:

    raise ValueError(
        "HIQ270 and HIQ210 missing positions differ."
    )


print(
    "PASS: HIQ270 and HIQ210 missing "
    "positions match exactly."
)


# ============================================================
# 32. VALID RESPONSE CODE CHECK
# ============================================================

VALID_CODES = {
    "HIQ011": {1, 2, 7, 9},

    "HIQ031A": {14, 77, 99},
    "HIQ031B": {15},
    "HIQ031C": {16},
    "HIQ031D": {17},
    "HIQ031E": {18},
    "HIQ031F": {19},
    "HIQ031G": {20},
    "HIQ031H": {21},
    "HIQ031I": {22},
    "HIQ031J": {23},
    "HIQ031AA": {40},

    "HIQ260": {1, 2, 9},
    "HIQ105": {1, 2},
    "HIQ270": {1, 2, 7, 9},
    "HIQ210": {1, 2, 9}
}


print(
    "\n--- VALID RESPONSE CODE CHECK ---"
)


for col, allowed_codes in VALID_CODES.items():

    observed_codes = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected = (
        observed_codes
        -
        allowed_codes
    )


    print(
        f"{col}:",
        sorted(
            observed_codes
        )
    )


    if unexpected:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{sorted(unexpected)}"
        )


print(
    "PASS: No unexpected HIQ response codes."
)


# ============================================================
# 33. CONFIRM NO ZEROS CREATED
# ============================================================

#
# Original file contained:
#
#     tiny zero artifacts = 0
#     ordinary zeros      = 0
#
#
# Therefore conversion must not create numeric zeros.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK BEFORE EXPORT ---"
)

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zero values were created:\n"
        + str(
            final_zero_counts
        )
    )


print(
    "PASS: No numeric zeros created."
)


# ============================================================
# 34. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# HIQ_I required NO numerical repair.
#
# Therefore every variable must remain numerically
# identical to the original XPT.
#
# Converting:
#
#     83732.0 -> 83732
#
# changes only text formatting in the CSV,
# not the numerical value.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 17 variables preserve "
    "source XPT numeric values exactly."
)


# ============================================================
# 35. EXPORT CSV
# ============================================================

#
# HIQ_I contains no genuine decimals.
#
# Still follow the standard high-fidelity export:
#
#     NO df.round(...)
#
#     NO float_format=
#
#
# This keeps the workflow safe and consistent with
# Dietary and other mixed continuous files.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 36. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 37. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 17 HIQ_I variables."
)


# ============================================================
# 38. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 39. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 40. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 41. RAW CSV .0 CHECK
# ============================================================

#
# HIQ_I contains NO genuine decimal fields.
#
# Therefore checking ALL 17 fields for unwanted ".0"
# is safe for THIS file.
#
#
# Examples:
#
#     SEQN
#         83732.0 -> 83732
#
#     HIQ011
#         1.0 -> 1
#
#     HIQ031A
#         14.0 -> 14
#
#     HIQ270
#         1.0 -> 1
#

print(
    "\n--- RAW CSV .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in any HIQ_I field."
)


# ============================================================
# 42. FINAL ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values found in final CSV:\n"
        + str(
            csv_zero_counts
        )
    )


print(
    "PASS: Final CSV contains no numeric zeros."
)


# ============================================================
# 43. FINAL FREQUENCY CHECK
# ============================================================

#
# High-value post-export checks.
#

FINAL_EXPECTED_FREQUENCIES = {

    "HIQ011": {
        1: 8640,
        2: 1312,
        7: 4,
        9: 15
    },

    "HIQ031A": {
        14: 4424,
        77: 4,
        99: 81
    },

    "HIQ031B": {
        15: 1371
    },

    "HIQ031D": {
        17: 2384
    },

    "HIQ260": {
        1: 101,
        2: 104,
        9: 7
    },

    "HIQ270": {
        1: 8147,
        2: 383,
        7: 3,
        9: 117
    },

    "HIQ210": {
        1: 674,
        2: 7965,
        9: 11
    }
}


print(
    "\n--- FINAL CSV FREQUENCY CHECK ---"
)


for col, expected_freq in FINAL_EXPECTED_FREQUENCIES.items():

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    actual_freq = (
        numeric_values
        .dropna()
        .astype(int)
        .value_counts()
        .sort_index()
        .to_dict()
    )


    print(
        f"{col}:",
        actual_freq
    )


    if actual_freq != expected_freq:

        raise ValueError(
            f"Final {col} frequencies changed."
        )


print(
    "PASS: Final key frequencies preserved."
)


# ============================================================
# 44. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 45. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 46. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "HIQ_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Questionnaire - Health Insurance"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original ordinary zeros:",
    true_zero_total
)


print(
    "Validated zero restorations:",
    "NONE"
)


print(
    "Genuine decimal variables:",
    "NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "Covered by health insurance:",
    int(
        df["HIQ011"]
        .eq(1)
        .sum()
    )
)


print(
    "Private insurance:",
    int(
        df["HIQ031A"]
        .eq(14)
        .sum()
    )
)


print(
    "Medicare coverage:",
    int(
        df["HIQ031B"]
        .eq(15)
        .sum()
    )
)


print(
    "Medicaid coverage:",
    int(
        df["HIQ031D"]
        .eq(17)
        .sum()
    )
)


print(
    "Prescription coverage follow-up:",
    int(
        df["HIQ270"]
        .notna()
        .sum()
    )
)


print(
    "Past-year insurance gap follow-up:",
    int(
        df["HIQ210"]
        .notna()
        .sum()
    )
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0':",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "HIQ_I contains no genuine decimal variables."
)


print(
    "All 17 variables can safely be exported "
    "as whole-number IDs/codes."
)


print(
    "Your broader NHANES fidelity rule remains unchanged:"
)


print(
    "Genuine Dietary and other continuous decimal "
    "variables remain exactly AS IS."
)

NHANES 2015-2016 HIQ_I XPT -> CSV
Rows: 9,971
Columns: 17
PASS: Structure = 9,971 rows x 17 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
String fields: []
Whole-number/code fields: ['SEQN', 'HIQ011', 'HIQ031A', 'HIQ031B', 'HIQ031C', 'HIQ031D', 'HIQ031E', 'HIQ031F', 'HIQ031G', 'HIQ031H', 'HIQ031I', 'HIQ031J', 'HIQ031AA', 'HIQ260', 'HIQ105', 'HIQ270', 'HIQ210']

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Variables containing tiny artifacts: 0
Ordinary numeric zeros: 0
PASS: HIQ_I contains 0 tiny artifacts and 0 ordinary numeric zeros.

PASS: No tiny-to-zero restoration required.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
HIQ011: 0 fractional observations
HIQ031A: 0 fractional observations
HIQ031B: 0 fractional observations
HIQ031C: 0 fractional observations
HIQ031D: 0 fractional observations
HIQ031E: 0 fractional observations
HIQ031F: 0 fractional observations
HIQ031G: 0 fractional observations
HIQ031H: 0 fractional observations
HIQ031